# Phase 2C5/C6 on Colab — completing the factorial (bouncer × ranker)

**What this is:** §9.5 (pre-registered before any run: `sanitycheck.md`, code `e8c04ac`, v8). C4-vs-C2 isolated the bouncer on BM25; this notebook fills the two empty cells of the ranker × bouncer design:

- **C5 = filtered_random** — C1's random supply + the bouncer. *Control:* the bouncer must be ≈ a no-op on a clean supply (F1 ≈ C1 0.7464; jaccard_filtered ≈ 0; audit ≈ 0–2 %). If C5 lands > 0.03 below C1: halt, something is unexplained.
- **C6 = filtered_semantic** — C3's semantic ranking (depth k×20) + the bouncer. *Discriminator* between **H1 similarity-trigger** (scorer de-saturates + F1 ≈/≥ C4's 0.43 band while contamination stays ~25–35 %) and **H2 dose-only** (C6 ≈ C3, pinned, ~0.27 — which OVERTURNS §9.4's reading and rewrites §6 around FN rate).

**Order:** §3 v8 re-mine (backs up C1–C4, asserts all four bit-identical — the audit_c4 sheet survives iff that passes) → §4 diagnostics (C4/C5/C6 rows) → §4.5 smoke → **§5–§6.5 TWO mandatory audit gates** → §7 campaign (n = 3 triage, seeds 13/14/15, both conditions; extend C6 to n = 6 iff within 0.05 of a decision boundary) → §8 consolidator → §9 GitHub. Budget: mining free (CPU, ~15 min); ~4 units GPU (n = 3 both), ~7–8 if both extend to n = 6.

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/ksu-gitreaper807/EmbedEd.git'
REPO_REF = 'arena/01a0e714-embeded'  # current Arena work branch (D1 gate rule, Phase 2 runner, Phase 3 runner); use main once merged
ROOT = '/content/EmbedEd'

def _git(*args, check=True):
    p = subprocess.run(['git', *args], cwd=ROOT, capture_output=True, text=True)
    if check and p.returncode:
        raise SystemExit(f"git {' '.join(args)} exited {p.returncode}\n{p.stdout}{p.stderr}")
    return p.stdout.strip()

if not os.path.isdir(os.path.join(ROOT, '.git')):
    subprocess.run(['git', 'clone', '--branch', REPO_REF, '--single-branch', REPO_URL, ROOT], check=True)
else:
    _git('fetch', 'origin', REPO_REF)
    ahead = _git('rev-list', '--count', 'FETCH_HEAD..HEAD', check=False)
    if ahead not in ('', '0'):
        raise SystemExit(
            f'{ahead} local commit(s) on this VM are not on origin/{REPO_REF}. Refusing to move the '
            'branch and orphan them: run notebook section 8 to push them (or `git rebase FETCH_HEAD` '
            'and push by hand), then re-run this cell.')
    _git('checkout', '-B', REPO_REF, 'FETCH_HEAD')
%cd /content/EmbedEd
print('Checked out:', _git('rev-parse', '--short', 'HEAD'))


## 1. Configure caches, install pinned dependencies, verify GPU, pull artifacts

Same environment as Phase 2C4.

In [ ]:
import os, sys
ROOT = '/content/EmbedEd'
ART = '/content/embeded-artifacts'
os.makedirs(ART, exist_ok=True)
os.environ['HF_HOME'] = '/content/hf-home'
os.environ['EMBEDED_ARTIFACTS'] = ART
os.environ['EMBEDED_REPORT'] = ROOT + '/report/measurements.md'
try:
    from google.colab import userdata
    for key in ('EMBEDED_HF_REPO_ID', 'EMBEDED_HF_REPO_TYPE', 'EMBEDED_HF_REVISION', 'EMBEDED_HF_SUBDIR', 'HF_TOKEN'):
        try: value = userdata.get(key)
        except Exception: value = None
        if value: os.environ.setdefault(key, value)
except Exception:
    pass
%pip install -q -U transformers==4.46.3 datasets==2.20.0 sentence-transformers==3.0.1 rank-bm25==0.2.2 umap-learn==0.5.6 scikit-learn==1.6.1 gradio==5.49.1 pytest==8.3.2 matplotlib==3.9.4
import torch, transformers, datasets, sentence_transformers, sklearn
print(f'torch={torch.__version__} cuda={torch.cuda.is_available()} | transformers={transformers.__version__} | datasets={datasets.__version__} | sklearn={sklearn.__version__}')
if not torch.cuda.is_available(): raise RuntimeError('GPU is required. Select a Colab T4 runtime before continuing.')
gpu = torch.cuda.get_device_name(0)
print('GPU:', gpu)
assert 'T4' in gpu.upper(), f'Expected the measured T4 configuration; got {gpu}. Re-measure throughput before using the frozen budget.'
from scripts.hf_artifacts import main as hf_main
# --light: trainers need fragments/triples/mining_summary, NOT the ~4.5 GB of checkpoints
hf_main(['pull', '--if-configured', '--light'])

## 2. Preflight

Expected state: `mining_summary.json` says `phase01-v7` (the C4 generation) and this checkout says **v8** — the §3 re-mine performs the sanctioned bump, with the bit-identity assert.

In [ ]:
from pathlib import Path
from embeded import settings as S
required = [S.artifact('fragments.jsonl'), S.artifact('corpus_emb.npy'), S.artifact('corpus_emb.meta.json')] + [S.artifact(f'triples_{c}.jsonl') for c in ('C1', 'C2', 'C3', 'C4')]
missing = [str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError('Missing artifacts; re-pull (cell above) first:\n' + '\n'.join(missing))
print('Artifacts present. Checkout VERSION =', S.VERSION)
import json as _json
_summary = S.ARTIFACTS / 'mining_summary.json'
_v = (_json.loads(_summary.read_text()).get('_run') or {}).get('version') if _summary.exists() else None
print('mining_summary.json _run.version =', _v)
if _v == S.VERSION:
    print('already v8 - section 3 prints [cache] and skips (delete triples_C5/C6.jsonl + edit the summary strategies to force)')
elif _v == 'phase01-v7':
    print('v7 artifacts: EXPECTED. Section 3 performs the sanctioned v8 re-mine (C1-C4 asserted bit-identical).')
else:
    raise SystemExit(f'unexpected mining version {_v!r} - stop and check with the notebook author.')

## 3. The sanctioned v8 re-mine — C1–C4 must come out bit-identical

Mines all six strategies under v8. **The assertion is the whole point:** C1–C4 triples must reproduce byte-for-byte (same seeds, unchanged code paths). If they do, the blind audit keys for C2/C3 *and* the fresh audit_c4 sheet all stay valid, and only C5/C6 are new bytes. If the assert ever fails: restore from `.v7bak` and stop — paste the output.

In [ ]:
import hashlib, json, os, shutil, subprocess, sys
from pathlib import Path
ROOT, ART = '/content/EmbedEd', '/content/embeded-artifacts'
os.chdir(ROOT)
if ROOT not in sys.path: sys.path.insert(0, ROOT)
os.environ.setdefault('EMBEDED_ARTIFACTS', ART)
os.environ.setdefault('EMBEDED_REPORT', ROOT + '/report/measurements.md')
from embeded import settings as S
import embeded.negatives as NG

def sha(p): return hashlib.sha256(open(p, 'rb').read()).hexdigest()

# --- staleness guard: this cell needs the C5/C6-era code (e8c04ac+) ---
head = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                      capture_output=True, text=True).stdout.strip()
print('checkout HEAD:', head, '| VERSION:', S.VERSION)
if not ('filtered_random' in NG.COND and 'filtered_semantic' in NG.COND
        and hasattr(S, 'C4_JACCARD_MAX')):
    raise SystemExit('STALE CHECKOUT: this clone predates the C5/C6 strategy (commit e8c04ac). '
                     'Re-run the checkout cell (section 0), then re-run THIS cell.')
print('this cell is the Phase2C5C6 v8 cell (streams output, tees a log).')

conds = ('C1', 'C2', 'C3', 'C4')
before = {}
for c in conds:
    t_path = S.artifact(f'triples_{c}.jsonl')
    bak = Path(str(t_path) + '.v7bak')
    if t_path.exists() and not bak.exists():   # keep the v7 originals once
        shutil.copy2(t_path, bak)
    if bak.exists():
        before[c] = sha(bak)
        print(f'backup on file: {bak.name} ({before[c][:12]}...)')

summary_path = S.ARTIFACTS / 'mining_summary.json'
run = ((json.loads(summary_path.read_text()).get('_run') or {})
       if summary_path.exists() else {})
if run.get('version') == S.VERSION and 'filtered_random' in run.get('strategies', []):
    print('[cache] mining_summary.json is already v8 with filtered_random - skipping.')
else:
    print('mining all six strategies (CPU; ~15 min)...', flush=True)
    log_path = Path('/content/mining_c5c6_log.txt')
    with open(log_path, 'w') as log:
        child = subprocess.Popen([sys.executable, '-u', '-m', 'embeded.negatives',
                                  '--strategies', 'random,bm25,semantic,filtered,filtered_random,filtered_semantic',
                                  '--force'],
                                 stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                 text=True, bufsize=1)
        for line in child.stdout:
            log.write(line)
            print(line, end='', flush=True)
        rc = child.wait()
    if rc:
        raise SystemExit(f'mining exited {rc}. Full log: {log_path} - print it with:\n'
                         f'  !tail -n 80 {log_path}\nand paste it verbatim into the session.')

changed = [c for c in conds if c in before and sha(S.artifact(f'triples_{c}.jsonl')) != before[c]]
if changed:
    raise SystemExit(
        f'RE-MINE CHANGED {changed}: the determinism assumption failed. The blind audit labels are now '
        f'orphaned. Restore:  for c in {" ".join(conds)}: mv artifacts/triples_$c.jsonl.v7bak artifacts/triples_$c.jsonl  '
        '(keep the new C5/C6 files), then re-label the audits before any training.')
if before:
    print('C1-C4 bit-identical to the v7 backups: ALL audit keys + labels survive (incl. audit_c4).')

summary = json.loads(summary_path.read_text())
assert (summary.get('_run') or {}).get('version') == S.VERSION, 'summary not v8?'
for s in ('filtered', 'filtered_random', 'filtered_semantic'):
    assert s in (summary.get('_run') or {}).get('strategies', []), f'{s} missing from summary'
for c in ('C5', 'C6'):
    print(c, summary[c]['triples'], 'triples |', summary[c]['stats'])
print('(C5 prediction: jaccard_filtered near 0. C6 prediction: a sizeable cut = C3\'s lexical sub-band.)')

## 4. Hardness diagnostics — the C4/C5/C6 rows

Runs automatically over every mined condition. Predictions: C5's mean cosine ≈ C1's row; C6 sits near C3 (same supply) minus its lexically-hot head. Paste the table.

In [ ]:
import os, subprocess, sys
ROOT, ART = '/content/EmbedEd', '/content/embeded-artifacts'
os.chdir(ROOT)
if ROOT not in sys.path: sys.path.insert(0, ROOT)
os.environ.setdefault('EMBEDED_ARTIFACTS', ART)
os.environ.setdefault('EMBEDED_REPORT', ROOT + '/report/measurements.md')
p = subprocess.run([sys.executable, '-u', '-m', 'scripts.hardness_diagnostics', '--examples', '2'])
if p.returncode: raise SystemExit(f'hardness_diagnostics exited {p.returncode} - reason printed above.')

## 4.5 Smoke test — C5 (~3 min on a T4)

Proves the new condition trains end-to-end. (C6 shares the same code path with a different ranker; if C5 smokes, run one C6 smoke too by changing the flag.)

In [ ]:
import os, subprocess, sys
ROOT, ART = '/content/EmbedEd', '/content/embeded-artifacts'
os.chdir(ROOT)
if ROOT not in sys.path: sys.path.insert(0, ROOT)
os.environ.setdefault('EMBEDED_ARTIFACTS', ART)
os.environ.setdefault('EMBEDED_REPORT', ROOT + '/report/measurements.md')
from importlib.util import find_spec
if find_spec('embeded.train') is None or find_spec('embeded.evaluate') is None:
    raise RuntimeError('runner modules missing from this checkout - check REPO_REF in the checkout cell.')
p = subprocess.run([sys.executable, '-u', '-m', 'embeded.train', '--condition', 'C5', '--seed', '13', '--smoke'])
if p.returncode:
    raise SystemExit(f'smoke run exited {p.returncode} - the reason is printed above. Do not start the campaign until this passes.')

## 5. AUDIT GATE 1 of 2 — 50-pair blind audit of C5

Blind sheet into `artifacts/audit_c5/` (no other audit is touched). Prediction: **≈ 0–2 % clones** (C1's band) — a random supply contains almost no false negatives, bouncer or not. Label offline, score in the next cell.

In [ ]:
import os, subprocess, sys
ROOT, ART = '/content/EmbedEd', '/content/embeded-artifacts'
os.chdir(ROOT)
if ROOT not in sys.path: sys.path.insert(0, ROOT)
os.environ.setdefault('EMBEDED_ARTIFACTS', ART)
os.environ.setdefault('EMBEDED_REPORT', ROOT + '/report/measurements.md')
os.environ['EMBEDED_AUDIT_DIR'] = ART + '/audit_c5'   # never touch the scored Phase-2 sheet
p = subprocess.run([sys.executable, '-u', '-m', 'scripts.audit_sample', 'make', '--conditions', 'C5', '--n', '50', '--force'])
if p.returncode: raise SystemExit(f'audit make exited {p.returncode} - reason printed above.')
from pathlib import Path
d = Path(ART) / 'audit_c5'
print()
print('sheet ready:', d / 'audit_pairs.md')
print('1. download audit_pairs.md from the file browser (left panel)')
print('2. for each pair decide: same functionality? label clone / not_clone / unsure')
print('3. fill audit_labels.csv (same file you downloaded the pairs from:', d / 'audit_labels.csv', ')')
print('4. do NOT open audit_key.csv')
print('5. upload the filled audit_labels.csv in section 5.5')

## 5.5 Score the C5 audit

In [ ]:
import json, os, shutil, subprocess, sys
from google.colab import files
ROOT, ART = '/content/EmbedEd', '/content/embeded-artifacts'
os.chdir(ROOT)
if ROOT not in sys.path: sys.path.insert(0, ROOT)
os.environ.setdefault('EMBEDED_ARTIFACTS', ART)
os.environ.setdefault('EMBEDED_REPORT', ROOT + '/report/measurements.md')
os.environ['EMBEDED_AUDIT_DIR'] = ART + '/audit_c5'

uploaded = files.upload()
if 'audit_labels.csv' not in uploaded:
    raise ValueError('Upload the completed file named audit_labels.csv')
shutil.move('audit_labels.csv', os.path.join(ART, 'audit_c5', 'audit_labels.csv'))

p = subprocess.run([sys.executable, '-u', '-m', 'scripts.audit_sample', 'score'])
if p.returncode: raise SystemExit(f'audit score exited {p.returncode} - reason printed above.')

res = json.loads(open(os.path.join(ART, 'audit_c5', 'audit_result.json')).read())
fn = res['C5']['fn_rate']; upper = res['C5']['fn_rate_upper_incl_unsure']
print()
print(f'=== AUDIT GATE: C5 clone rate {fn:.0%} (upper bound incl. unsure {upper:.0%}) ===')
print('    references: C1 ~0-2% | C2 42% [29, 56] | gate: <= 42%, prediction 0-2%')
if fn > 0.42:
    raise SystemExit('GATE FAILED: C4 contamination is at or above the C2 floor. Training may proceed '
                     'but every C4 comparison is footnoted as "denoising attempted, contamination '
                     'persists" (pre-registration 9.4). Paste this output to the session.')
print('GATE PASSED. C4 comparisons may be made on equal footing once the campaign finishes.')

## 6. AUDIT GATE 2 of 2 — 50-pair blind audit of C6

Blind sheet into `artifacts/audit_c6/`. Prediction under **H1**: ~25–35 % (the semantic FN mass passes the lexical cut — HIGH contamination is *expected* and does NOT fail the gate; the gate bound stays ≤ 42 %). Under **H2** the rate could be lower — the rate alone does not decide H1 vs H2; the C6 F1/threshold behaviour does (that's §7.5's reading).

In [ ]:
import os, subprocess, sys
ROOT, ART = '/content/EmbedEd', '/content/embeded-artifacts'
os.chdir(ROOT)
if ROOT not in sys.path: sys.path.insert(0, ROOT)
os.environ.setdefault('EMBEDED_ARTIFACTS', ART)
os.environ.setdefault('EMBEDED_REPORT', ROOT + '/report/measurements.md')
os.environ['EMBEDED_AUDIT_DIR'] = ART + '/audit_c6'   # never touch the scored Phase-2 sheet
p = subprocess.run([sys.executable, '-u', '-m', 'scripts.audit_sample', 'make', '--conditions', 'C6', '--n', '50', '--force'])
if p.returncode: raise SystemExit(f'audit make exited {p.returncode} - reason printed above.')
from pathlib import Path
d = Path(ART) / 'audit_c6'
print()
print('sheet ready:', d / 'audit_pairs.md')
print('1. download audit_pairs.md from the file browser (left panel)')
print('2. for each pair decide: same functionality? label clone / not_clone / unsure')
print('3. fill audit_labels.csv (same file you downloaded the pairs from:', d / 'audit_labels.csv', ')')
print('4. do NOT open audit_key.csv')
print('5. upload the filled audit_labels.csv in section 5.5')

## 6.5 Score the C6 audit

In [ ]:
import json, os, shutil, subprocess, sys
from google.colab import files
ROOT, ART = '/content/EmbedEd', '/content/embeded-artifacts'
os.chdir(ROOT)
if ROOT not in sys.path: sys.path.insert(0, ROOT)
os.environ.setdefault('EMBEDED_ARTIFACTS', ART)
os.environ.setdefault('EMBEDED_REPORT', ROOT + '/report/measurements.md')
os.environ['EMBEDED_AUDIT_DIR'] = ART + '/audit_c6'

uploaded = files.upload()
if 'audit_labels.csv' not in uploaded:
    raise ValueError('Upload the completed file named audit_labels.csv')
shutil.move('audit_labels.csv', os.path.join(ART, 'audit_c6', 'audit_labels.csv'))

p = subprocess.run([sys.executable, '-u', '-m', 'scripts.audit_sample', 'score'])
if p.returncode: raise SystemExit(f'audit score exited {p.returncode} - reason printed above.')

res = json.loads(open(os.path.join(ART, 'audit_c6', 'audit_result.json')).read())
fn = res['C6']['fn_rate']; upper = res['C6']['fn_rate_upper_incl_unsure']
print()
print(f'=== AUDIT GATE: C6 clone rate {fn:.0%} (upper bound incl. unsure {upper:.0%}) ===')
print('    references: C3 30% [19, 44] | C2 42% | gate: <= 42%, H1 prediction 25-35%')
if fn > 0.42:
    raise SystemExit('GATE FAILED: C4 contamination is at or above the C2 floor. Training may proceed '
                     'but every C4 comparison is footnoted as "denoising attempted, contamination '
                     'persists" (pre-registration 9.4). Paste this output to the session.')
print('GATE PASSED. C4 comparisons may be made on equal footing once the campaign finishes.')

## 7. The campaign — C5 × seeds 13/14/15 + C6 × seeds 13/14/15 (n = 3 triage)

Six slots, ~4 GPU units. Edit `RUNS` per VM (two VMs: C5 on one, C6 on the other). Extend a condition to n = 6 (seeds 16/17/18) ONLY if its F1 lands within 0.05 of a decision boundary: C5 within 0.03 of C1, or C6 between the H1 band (≥ ~0.40) and the H2 band (≤ ~0.32). Interrupt-after-push safe; never restart a finished run.

In [ ]:
import os, subprocess, sys
ROOT, ART = '/content/EmbedEd', '/content/embeded-artifacts'
os.chdir(ROOT)
if ROOT not in sys.path: sys.path.insert(0, ROOT)
os.environ.setdefault('EMBEDED_ARTIFACTS', ART)
os.environ.setdefault('EMBEDED_REPORT', ROOT + '/report/measurements.md')
os.environ['PYTHONPATH'] = ROOT + os.pathsep + os.environ.get('PYTHONPATH', '')
from scripts.hf_artifacts import main as hf_main

# ---- Phase 2C4 campaign: six seeds on the frozen v7 triples -------------------
ALL_RUNS = [('C5', s) for s in (13, 14, 15)] + [('C6', s) for s in (13, 14, 15)]

# >>> THIS VM's slots. Edit before running. <<<
RUNS = ALL_RUNS            # <-- EDIT ME (default = this VM would do all six)

print('this VM will run:', RUNS)
assert len(set(RUNS)) == len(RUNS), 'duplicate slots in this VM list'
assert set(RUNS) <= set(ALL_RUNS), 'slots outside the campaign'

# preflight: v8 summary with 'filtered_random' present, C5/C6 triples on disk
import json
from embeded import settings as S
summary = json.loads((S.ARTIFACTS / 'mining_summary.json').read_text())
run = summary.get('_run') or {}
if run.get('version') != S.VERSION or 'filtered_random' not in run.get('strategies', []):
    raise SystemExit('stale artifacts: run section 3 (the sanctioned v8 re-mine) first')
for c in ('C5', 'C6'):
    if not (S.ARTIFACTS / f'triples_{c}.jsonl').exists():
        raise SystemExit(f'missing triples_{c}.jsonl - run section 3 first')

def sh(*args):
    print('$ python -m ' + ' '.join(args), flush=True)
    rc = subprocess.run([sys.executable, '-u', '-m', *args]).returncode
    if rc:
        raise SystemExit(f'`python -m {args[0]}` exited {rc} - reason printed above.')

for condition, seed in RUNS:
    print(f'\n===== TRAIN {condition} seed={seed} =====', flush=True)
    sh('embeded.train', '--condition', condition, '--seed', str(seed))
    print(f'===== EVALUATE {condition} seed={seed} =====', flush=True)
    sh('embeded.evaluate', '--condition', condition, '--seed', str(seed))
    hf_main(['push', '--if-configured'])   # run dir only - NEVER --include-report from a trainer
print('\nCampaign slots done:', RUNS)
print('Re-running this cell is safe: finished runs skip.')

## 8. After this VM's slots finish

Push the run dirs once more (idempotent) and stop — unless this is the consolidating VM: then also §8.5.

In [ ]:
from scripts.hf_artifacts import main as hf_main
# Trainer VMs push ARTIFACTS ONLY. The report is rebuilt once, on the consolidating VM.
hf_main(['push', '--if-configured'])
print('run dirs synced. Report/GitHub steps belong to the consolidating VM (next section).')

## 8.5 Consolidating VM only — rebuild the report from the union

The board now auto-discovers C0–C6; the table gains the C5/C6 rows.

In [ ]:
import os, subprocess, sys
from pathlib import Path
ROOT, ART = '/content/EmbedEd', '/content/embeded-artifacts'
os.chdir(ROOT)
if ROOT not in sys.path: sys.path.insert(0, ROOT)
os.environ.setdefault('EMBEDED_ARTIFACTS', ART)
os.environ.setdefault('EMBEDED_REPORT', ROOT + '/report/measurements.md')
os.environ['PYTHONPATH'] = ROOT + os.pathsep + os.environ.get('PYTHONPATH', '')
from scripts.hf_artifacts import main as hf_main

hf_main(['pull', '--if-configured', '--light'])

p = subprocess.run([sys.executable, '-u', '-m', 'embeded.evaluate', '--results-table'])
if p.returncode:
    raise SystemExit('--results-table failed - reason printed above.')
print('table now aggregates every discovered run (C0-C3 bands + the new C4 rows).')

import pathlib
from scripts.plot_confusion_matrices import main as cm_main
_runs = pathlib.Path(ART) / 'runs'
_out  = pathlib.Path(ROOT) / 'report' / 'figures' / 'phase2_confusion_matrices.png'
cm_main(['--runs-dir', str(_runs), '--out', str(_out)])

hf_main(['push', '--if-configured', '--include-report'])
print('consolidation pushed. Now the GitHub step (final cell).')

## 9. Push the VM's commits back to GitHub

In [ ]:
import json, os, subprocess, sys, urllib.error, urllib.request

ROOT = '/content/EmbedEd'
REF  = REPO_REF                 # cell 2
GIT_NAME  = ''                  # your GitHub username; '' = derive it from the PAT below
GIT_EMAIL = ''                  # the noreply address shown at https://github.com/settings/emails
COMMIT_DIRTY_REPORT = True      # commit tracked changes under report/ before rebasing
TOKEN_SECRET = 'GITHUB_TOKEN'   # Colab Secrets panel (key icon). No token -> stops before pushing.
API_USER = 'https://api.github.com/user'

def sh(*args, check=False):
    env = {k: v for k, v in os.environ.items()
           if not k.startswith(('GIT_AUTHOR_', 'GIT_COMMITTER_'))}   # these override user.email
    p = subprocess.run(['git', '-C', ROOT, *args], capture_output=True, text=True, env=env)
    shown = (p.stdout + p.stderr).strip()
    print(f"$ git {' '.join(args)} -> exit {p.returncode}")
    if shown:
        print('  ' + shown.replace('\n', '\n  '))
    if check and p.returncode:
        raise SystemExit(f'git {" ".join(args)} failed -- stopping, nothing was lost')
    return p.returncode, p.stdout          # raw: porcelain needs its leading status column

def read_token():                          # Colab Secrets first, then the environment
    try:
        from google.colab import userdata
        value = userdata.get(TOKEN_SECRET)
        if value:
            return value.strip()
    except Exception:
        pass
    return (os.environ.get(TOKEN_SECRET) or '').strip()

# 1. identity first: every commit and every rebase needs one, and a fresh VM has none
if not sh('config', '--get', 'user.email')[1].strip():
    name, email = GIT_NAME, GIT_EMAIL
    token = read_token()
    if not (name and email) and token:
        req = urllib.request.Request(API_USER, headers={'Authorization': f'Bearer {token}',
                                                        'Accept': 'application/vnd.github+json',
                                                        'User-Agent': 'embeded-colab'})
        try:
            with urllib.request.urlopen(req, timeout=20) as resp:
                info = json.loads(resp.read().decode())
            if info.get('id') and info.get('login'):
                name = info['login']
                email = f"{info['id']}+{info['login']}@users.noreply.github.com"
                print(f"  derived the identity from the PAT: {name} <{email}>")
        except (urllib.error.URLError, urllib.error.HTTPError, OSError, ValueError) as exc:
            print(f'  could not reach {API_USER}: {exc.__class__.__name__}')
    if not (name and email):
        raise SystemExit(
            'No git identity on this VM and nothing to derive one from. Pick one:\n'
            f'  a) set GIT_NAME / GIT_EMAIL at the top of this cell, or\n'
            f'  b) add the PAT as a Colab Secret named {TOKEN_SECRET} (then it is derived above), or\n'
            f'  c) run:  !git -C {ROOT} config user.name "you"  and  '
            f'!git -C {ROOT} config user.email "ID+you@users.noreply.github.com"\n'
            'Nothing was changed; re-run this cell afterwards.')
    sh('config', 'user.name', name, check=True)
    sh('config', 'user.email', email, check=True)
    print('  commits will now be authored by: {} <{}>'.format(
        sh('config', '--get', 'user.name')[1].strip(),
        sh('config', '--get', 'user.email')[1].strip()))

# 2. tracked changes block a rebase; untracked strays usually do not
_, raw = sh('status', '--porcelain=v1')
entries = [(line[:2], line[3:].strip()) for line in raw.splitlines() if line.strip()]
tracked = [path for xy, path in entries if xy.strip() != '??']
untracked = [path for xy, path in entries if xy.strip() == '??']

# 3. commit tracked changes under report/ only; audit_key* is never staged
report = [p for p in tracked if p.startswith('report/') and 'audit_key' not in p]
other = [p for p in tracked if p not in report]
if other:
    raise SystemExit(f'tracked changes outside report/ {other} -- commit or stash them, then re-run')
if tracked:
    if not COMMIT_DIRTY_REPORT:
        raise SystemExit(f'tracked changes {tracked} but COMMIT_DIRTY_REPORT is False')
    sh('add', '--', *report, check=True)
    sh('commit', '-m', 'report: VM measurements + audit separability', check=True)

# 4. public repo: fetch needs no token
sh('fetch', 'origin', REF, check=True)

# 5. an untracked file only blocks the rebase if the remote adds that same path
for path in untracked:
    if sh('ls-tree', '--name-only', 'FETCH_HEAD', '--', path)[1].strip():
        raise SystemExit(f'{path} is untracked here but exists on the remote -- move or delete it, '
                         'then re-run')
    print(f'  (untracked, remote does not have it -> leaving it alone: {path})')

# 6. replay any local commit on top of the branch tip
if sh('rebase', 'FETCH_HEAD')[0]:
    raise SystemExit('rebase stopped -- your commits are safe. Either resolve the conflict and '
                     '`git rebase --continue`, or `git -C %s rebase --abort` to undo it.' % ROOT)
sh('log', '--oneline', '-4')

# 7. scripts.colab_git only exists after the rebase above
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
sys.modules.pop('scripts.colab_git', None)  # a cached pre-rebase copy would shadow the new code
from scripts.colab_git import diagnose, read_secret, sync_and_push

print('\n--- diagnose ---')
for line in diagnose(ROOT, REF):
    print(line)

token = read_secret((TOKEN_SECRET,))
if not token:
    print(f'\nNo {TOKEN_SECRET} in Colab Secrets -> stopping before the push.')
    print('Add the fine-grained PAT as a Colab Secret named GITHUB_TOKEN, then re-run this cell:')
    print('the identity, the commit and the rebase are already done, so it goes straight to push.')
else:
    print('\n--- push ---')
    result = sync_and_push(ROOT, REF, token=token)
    for line in result['lines']:
        print(line)
    print(('ok  ' if result['ok'] else 'FAIL') + f" [{result['stage']}] {result['message']}")


## 10. Reading the results (decision rules fixed in §9.5)

Paste back, in order: (1) §3's re-mine — the C1–C4-identical line + C5/C6 mining stats (C5 jaccard_filtered ≈ 0? C6's cut size?); (2) §4's diagnostics rows; (3) both gate blocks; (4) per-slot completions; (5) §8.5's table.

Verdict logic: **C5** — F1 within C1's band + audit ≈ 0–2 % ⇒ bouncer validated as band-removal (control passed); C5 ≫ 0.03 below C1 ⇒ halt and investigate. **C6** — read the primary endpoint first: thresholds pinned ⇒ H2 (dose-only; §9.4's similarity-axis claim OVERTURNED, §6 rewritten); thresholds unpinned ⇒ H1 survives, then F1 places the residual: ≥ C4 band with contamination 25–35 % ⇒ "lexical denoising de-saturates but semantic FN mass caps quality" (the §9.4 refinement confirmed within-ranker); unpinned + contamination low ⇒ similarity-trigger strengthened further. No comparison is headlined before BOTH gates are scored.